<a href="https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

## 1. Unit of analysis + time window

Unit of Analysis (Grain): One row = one page-day (a specific content_hash_id for a specific client_hash_id on a specific report_date).

Time Window: The observation window is March 2026 (filtered by the month column partition). We are avoiding the sealed final month.

Target Table: fact_content_daily_performance from FlyRank/internship-warehouse.

Target / Proxy: is_declining_proxy. Because this is raw daily data, we derive a proxy target (gsc_clicks == 0) to flag pages failing to capture search traffic.

Output: A ranked scoring of page-days to help identify underperforming content.

In [14]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

# Path to the daily performance parquets
WAREHOUSE_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

## 2. Fields: feature / label / context / excluded

Context: content_hash_id, client_hash_id, report_date, month. Used for grouping; never for the model.

Label / Proxy: is_declining_proxy (Derived as 1 if gsc_clicks == 0, else 0).

Features:

1) gsc_impressions: Knowable at the decision moment because Search Console logs finalize daily.

2) gsc_avg_position: Knowable at the decision moment via search logs (we replace 0 with 100.0 to fix missing ranks).

3) ga4_sessions: Knowable at the decision moment via daily analytics exports.

4) ga4_total_engagement_sec: Knowable at the decision moment via active session timers.

5) sessions_ai: Knowable at the decision moment via AI user-agent log aggregation.

Excluded: gsc_clicks must be deliberately excluded as a feature (it is the trap). Because our proxy label is derived from it, including it causes target leakage.

In [15]:
feature_query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    -- Proxy Target
    CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END AS is_declining_proxy,

    -- 5 Features
    gsc_impressions,
    CASE WHEN gsc_avg_position = 0 THEN 100.0 ELSE gsc_avg_position END AS adjusted_avg_position,
    ga4_sessions,
    ga4_total_engagement_sec,
    sessions_ai,

    -- TRAP FEATURE: Label-derived column
    gsc_clicks AS LEAK_clicks
FROM read_parquet('{WAREHOUSE_PATH}')
WHERE month = '2026-03'
  AND ga4_data_available IS TRUE
"""
df_slice = con.execute(feature_query).df()
print(f"Feature frame loaded: {df_slice.shape[0]} rows, {df_slice.shape[1]} columns.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame loaded: 413966 rows, 10 columns.


## 3. Verify it with queries (grain, counts, missing values, windows)

*This section runs the three verification checks for grain, slice boundaries, and data availability, followed by the deliberate leakage trap experiment.*

In [16]:
# 1. Grain Verification (Should return 0 rows if unit holds as page-day)
grain_check = con.execute(f"""
SELECT content_hash_id, client_hash_id, report_date, COUNT(*) as c
FROM read_parquet('{WAREHOUSE_PATH}')
WHERE month = '2026-03'
GROUP BY content_hash_id, client_hash_id, report_date
HAVING c > 1
LIMIT 5
""").df()
print(f"Grain Check (duplicates): {len(grain_check)}")

# 2. Slice Bounds Verification (Counts and windows)
span_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{WAREHOUSE_PATH}')
WHERE month = '2026-03'
""").df()
print(f"\nSlice Bounds:\n{span_check}")

# 3. Availability Verification (ga4_data_available)
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS total_slice_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS verified_ga4_rows
FROM read_parquet('{WAREHOUSE_PATH}')
WHERE month = '2026-03'
""").df()
print(f"\nAvailability Check:\n{availability_check}")

# --- 4. The Leakage Trap Experiment ---
print("\n--- Leakage Trap Check ---")
df_clean = df_slice.dropna().copy()
features_clean = ['gsc_impressions', 'adjusted_avg_position', 'ga4_sessions', 'ga4_total_engagement_sec', 'sessions_ai']
features_leaky = features_clean + ['LEAK_clicks']
y = df_clean['is_declining_proxy'].astype(int)

# Leaky Model (Using gsc_clicks to predict a target derived from gsc_clicks)
clf_leaky = LogisticRegression(max_iter=1000)
clf_leaky.fit(df_clean[features_leaky], y)
preds_leaky = clf_leaky.predict_proba(df_clean[features_leaky])[:, 1]
print(f"Leaky Model ROC-AUC: {roc_auc_score(y, preds_leaky):.4f} (Perfect score due to target leakage)")

# Honest Model (Using only prior/independent signals)
clf_clean = LogisticRegression(max_iter=1000)
clf_clean.fit(df_clean[features_clean], y)
preds_clean = clf_clean.predict_proba(df_clean[features_clean])[:, 1]
print(f"Honest Model ROC-AUC: {roc_auc_score(y, preds_clean):.4f} (Realistic baseline)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain Check (duplicates): 0

Slice Bounds:
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

Availability Check:
   total_slice_rows  verified_ga4_rows
0           9841378             413966

--- Leakage Trap Check ---
Leaky Model ROC-AUC: 1.0000 (Perfect score due to target leakage)
Honest Model ROC-AUC: 0.7580 (Realistic baseline)


## 4. Data limits

Named Limitation: Because we require ga4_data_available IS TRUE, our model is blind to clients whose GA4 infrastructure broke or was improperly implemented during this month. It assumes missing GA4 data means "no traffic," when in reality, the tracking pipeline might just be disconnected.

In [18]:
# Validating missingness patterns to highlight limits
null_check = df_slice[['gsc_impressions', 'adjusted_avg_position', 'ga4_sessions', 'ga4_total_engagement_sec', 'sessions_ai']].isnull().mean() * 100
print("Percentage of missing values per column in our available slice:")
print(null_check.round(2).astype(str) + '%')

Percentage of missing values per column in our available slice:
gsc_impressions               0.0%
adjusted_avg_position       11.99%
ga4_sessions                  0.0%
ga4_total_engagement_sec      0.0%
sessions_ai                   0.0%
dtype: object


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.